# Capstone: train a small reasoner with RL (T2 path)

One T4, Qwen3-0.6B-Base, GRPO on Countdown. Run the cells from the top. Each session trains one cell of the ablation until `MAX_MINUTES`, saves, and stops. Run the notebook again to continue.

Kaggle: Settings > Accelerator > GPU T4, and Internet on. Colab: Runtime > Change runtime type > T4 GPU.

<!-- llt-run-help -->
## How to run this notebook

- **Check the lab card** on the module page first. It says whether this lab needs a GPU and how long it took us.
- **Kaggle** (where we recorded most runs): use the lab page's *Open on Kaggle* button. If the lab needs a GPU, open the notebook settings and pick *Accelerator: GPU T4 x2* (not P100). Switch *Internet* on so the install cell and the downloads work. Kaggle asks you to verify a phone number once before it allows a GPU or internet.
- **Colab**: use the lab page's *Open in Colab* button. If the lab needs a GPU, pick *Runtime > Change runtime type > T4 GPU*.
- **Hugging Face**: no account or token is needed. Our checkpoints and data files download from the public repo [aimlcompanion/llm-training-lab](https://huggingface.co/aimlcompanion/llm-training-lab).
- **Run the cells in order.** Code cell numbers match the lab page. Your numbers should land close to ours, not match exactly: GPU runs are not bit-for-bit repeatable.

In [ ]:
import os, sys, shutil, subprocess
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
assert sys.version_info >= (3, 11), 'Python 3.11 or newer is needed'
try:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print('No GPU found. Runtime > Change runtime type > T4 GPU (Colab), or Accelerator > GPU T4 (Kaggle)')
print('free disk GB', round(shutil.disk_usage('.').free / 1e9, 1))

In [ ]:
!pip uninstall -y -q torchao
!pip install -q transformers==5.22.1 trl==1.17.1 peft==0.21.2 accelerate==1.15.0 datasets==5.1.0 math-verify==0.9.0

In [ ]:
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
os.chdir(WORK)
if not os.path.isdir('aiml-companion'):
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/genieincodebottle/aiml-companion.git
    !cd aiml-companion && git sparse-checkout set courses/llm-training/labs/lltCapstoneReasoner courses/llm-training/labs/lltRL
LAB = os.path.join(WORK, 'aiml-companion/courses/llm-training/labs/lltCapstoneReasoner')
os.chdir(LAB)
!python test_capstone.py

## Choose the cell this session trains

The ablation has 4 cells x 3 seeds (our reference ran 3 seeds per cell). Keep `OUT` on storage that survives the session (Kaggle: save the output as a dataset or download `runs/`; Colab: mount Google Drive and point `RUNS` there).

In [ ]:
AGG, SCALE, SEED = 'grpo', 1, 0          # agg: grpo | dr_grpo ; scale: 1 | 0 ; seed: 0 | 1
MAX_MINUTES = 100                         # stop cleanly before the session limit, then resume
RUNS = os.path.join(WORK, 'runs')
NAME = 'agg-%s_scale-%d_s%d' % (AGG, SCALE, SEED)
OUT = os.path.join(RUNS, NAME)
print(OUT)

In [ ]:
# Base model first (once): pass@1 to pass@16 on the 200 held-out problems
if not os.path.exists(os.path.join(WORK, 'evals/base.json')):
    !python evaluate.py --out {WORK}/evals/base.json --n 16 --batch 64

In [ ]:
!python train.py --out {OUT} --agg {AGG} --scale {SCALE} --seed {SEED} --max_minutes {MAX_MINUTES}

In [ ]:
# After the run has finished (done.json exists), evaluate it
if os.path.exists(os.path.join(OUT, 'done.json')):
    !python evaluate.py --out {WORK}/evals/{NAME}.json --adapter {OUT}/adapter-final --n 16 --batch 64
else:
    print('Not finished yet. Run the notebook again to continue training.')

In [ ]:
# When all 8 evaluations exist
!python analyze.py --runs {RUNS} --evals {WORK}/evals --out {WORK}/results.json